# 07 - Analisis de estacionalidad mensual
Insumo: data/processed/suicidio_chihuahua_2019_2024.csv (output de
01_filter_chihuahua.ipynb, ya filtrado por Ent_resid == '08').

No se necesita denominador poblacional aqui: es una distribucion de
conteos por mes, no una tasa. Por eso se parte directo del output de
01, sin pasar por el merge de poblacion de 02.

OBJETIVO: probar si los casos de suicidio en Chihuahua (2019-2024) se
distribuyen uniformemente entre los 12 meses del anio, o si hay meses
con exceso, y comparar contra el patron reportado por Fernandez-Lopez
et al. (2021) para Chihuahua 2008-2018 (picos en junio y julio,
p < .002, asociado a temperatura).

NOTA METODOLOGICA: Fernandez-Lopez et al. usaron un modelo de series de
tiempo con temperatura ambiental como covariable. Aqui se usa una
prueba chi-cuadrado de bondad de ajuste (uniformidad mensual) -- un
primer acercamiento mas simple, no un modelo estacional equivalente.
Documentar esto como limitacion en el articulo si se usa este resultado.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
from cleaning_utils import test_estacionalidad_mensual, MESES_NOMBRE

df_chih = pd.read_csv(
    '../data/processed/suicidio_chihuahua_2019_2024.csv',
    encoding='utf-8', low_memory=False, dtype=str
)
print(f'Registros de Chihuahua cargados: {len(df_chih):,}')


## 1. Prueba de uniformidad mensual, agregado 2019-2024

In [ ]:
resultado = test_estacionalidad_mensual(df_chih, columna_mes='Mes_ocurr')

print('Calidad de datos (columna Mes_ocurr):')
for k, v in resultado['calidad_datos'].items():
    print(f'  {k}: {v:,}')
print()
print(f"chi2 = {resultado['chi2']}, p-value = {resultado['p_value']:.4g}")
print(f"Meses con mayor exceso sobre lo esperado: {resultado['meses_pico']}")
print()
resultado['tabla_mensual']


## 2. Estabilidad del patron por anio
Mismo enfoque que se uso para el RR de la Sierra Tarahumara (notebook
06): antes de confiar en el patron agregado 2019-2024, verificar que no
sea un artefacto de un solo anio atipico.

In [ ]:
resultados_por_anio = {}
for anio in sorted(df_chih['anio_dataset'].unique()):
    df_anio = df_chih[df_chih['anio_dataset'] == anio]
    r = test_estacionalidad_mensual(df_anio, columna_mes='Mes_ocurr')
    resultados_por_anio[anio] = r
    print(f"{anio}: chi2={r['chi2']}, p={r['p_value']:.4g}, "
          f"meses pico={r['meses_pico']}")


## 3. Comparacion contra Fernandez-Lopez et al. (2021)
Referencia: picos ciclicos en primavera-verano, mayores registros en
junio y julio, estacionalidad significativa en ambos sexos (p < .002),
periodo 2008-2018.

_Completar tras correr las celdas anteriores:_
- ?Los meses pico 2019-2024 coinciden con junio-julio? ?Si, no, o
  parcialmente?
- ?El patron se mantiene estable en los 6 anios o solo aparece en
  algunos?
- ?El p-value agregado es significativo? (ojo: p pequeño no implica que
  el efecto sea grande -- revisar tambien el exceso_pct de la tabla).

## 4. Guardar tabla mensual

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)
resultado['tabla_mensual'].to_csv(
    '../data/processed/estacionalidad_mensual_chihuahua_2019_2024.csv',
    index=False, encoding='utf-8'
)
print('Guardado: estacionalidad_mensual_chihuahua_2019_2024.csv')


## 5. Hallazgos
_Completar con los resultados reales tras correr el notebook:_
- chi2, p-value y meses pico del agregado 2019-2024.
- Estabilidad (o no) del patron anio con anio.
- Comparacion explicita contra Fernandez-Lopez et al. (2021): ?se
  sostiene el pico de junio-julio 11 anios despues, o cambio?
- Registros con mes no especificado o nulo excluidos del analisis
  (ver 'calidad_datos' arriba) -- documentar en methodology.md si es
  una proporcion no trivial.